In [ ]:
# ============================================================
# 04_convert_to_gguf.py
# 목적:
#   merged Hugging Face 모델을 GGUF로 변환
# ============================================================

!git clone https://github.com/ggerganov/llama.cpp.git /content/llama.cpp
!pip -q install -r /content/llama.cpp/requirements.txt

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/Army_Finetune")
MERGED_DIR = BASE_DIR / "03_outputs" / "army_merged_model"
GGUF_DIR = BASE_DIR / "03_outputs" / "army_gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)

FP16_GGUF = GGUF_DIR / "army-finetuned-f16.gguf"
Q4_GGUF = GGUF_DIR / "army-finetuned-q4_k_m.gguf"


# ------------------------------------------------------------
# 1. HF 모델 → F16 GGUF
# ------------------------------------------------------------
!python /content/llama.cpp/convert_hf_to_gguf.py \
  "{MERGED_DIR}" \
  --outfile "{FP16_GGUF}" \
  --outtype f16


# ------------------------------------------------------------
# 2. F16 GGUF → Q4_K_M 양자화
# ------------------------------------------------------------
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DLLAMA_CURL=OFF
!cmake --build /content/llama.cpp/build --config Release -j 2

! /content/llama.cpp/build/bin/llama-quantize \
  "{FP16_GGUF}" \
  "{Q4_GGUF}" \
  Q4_K_M

print("GGUF 변환 완료:")
print(Q4_GGUF)

FROM ./army-finetuned-q4_k_m.gguf

PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 8192
PARAMETER num_predict 1000

SYSTEM """
너는 미 육군 교리 기반 RAG 답변 보조자다.
반드시 제공된 검색 근거 범위 안에서만 답변한다.

답변은 다음 5개 항목을 모두 포함해야 한다.

1. 질문 해석
2. 관련 교범 근거
3. 작전적 검토
4. 권고 또는 참모 검토사항
5. 한계사항

답변 본문에는 chunk_id, chunk, 청크 같은 내부 데이터 관리 용어를 절대 사용하지 않는다.
교범명, 장·절, 페이지, 핵심 근거만 사용자가 이해할 수 있는 방식으로 제시한다.
근거가 부족하면 단정하지 말고, 검색된 자료 범위와 추가 확인 필요성을 명시한다.
실제 작전 실행명령, 특정 표적 타격 우선순위, 민감한 세부 전술은 일반 교리 원칙과 참모 검토사항 수준으로 제한한다.
"""

# ============================================================
# 05_compare_base_vs_finetuned.py
# 목적:
#   llama3.1:8b와 army-finetuned:latest 답변 비교
#
# 필요:
#   로컬에서 Ollama 실행 중이어야 함
# ============================================================

import requests
import time
import re
import pandas as pd
from pathlib import Path


OLLAMA_URL = "http://localhost:11434/api/generate"

MODELS = [
    "llama3.1:8b",
    "army-finetuned:latest",
]

QUESTIONS = [
    "작전계획을 수립할 때 지휘관과 참모가 공통적으로 고려해야 할 핵심 사항은 무엇인가?",
    "그렇다면 작전계획을 실제 타격·화력 운용계획으로 구체화할 때 지휘관과 참모가 검토해야 할 사항은 무엇인가?",
    "표적처리 또는 targeting 절차는 타격·화력 운용계획과 어떤 관계가 있으며, 지휘관과 참모는 어떤 순서로 이를 검토해야 하는가?",
]

SYSTEM_PROMPT = """너는 미 육군 교리 기반 RAG 답변 보조자다.
반드시 제공된 검색 근거 범위 안에서만 답변한다.
답변은 다음 5개 항목을 모두 포함해야 한다.

1. 질문 해석
2. 관련 교범 근거
3. 작전적 검토
4. 권고 또는 참모 검토사항
5. 한계사항

답변 본문에는 chunk_id, chunk, 청크 같은 내부 데이터 관리 용어를 절대 사용하지 않는다.
근거가 부족하면 단정하지 말고, 검색된 자료 범위와 추가 확인 필요성을 명시한다."""


def build_prompt(question):
    return f"""{SYSTEM_PROMPT}

[질문]
{question}

위 질문에 대해 교리 기반으로 답변하라.
답변은 반드시 다음 형식을 따른다.

1. 질문 해석
2. 관련 교범 근거
3. 작전적 검토
4. 권고 또는 참모 검토사항
5. 한계사항
"""


def call_ollama(model, prompt):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.2,
            "top_p": 0.9,
            "repeat_penalty": 1.1,
            "num_ctx": 8192,
            "num_predict": 1200,
        },
    }

    start = time.time()
    response = requests.post(OLLAMA_URL, json=payload, timeout=1800)
    elapsed = time.time() - start

    response.raise_for_status()
    data = response.json()

    return data.get("response", ""), elapsed


def count_citations(answer):
    patterns = [
        r"\bFM\s?\d",
        r"\bADP\s?\d",
        r"\bATP\s?\d",
        r"\bAFDP\s?\d",
        r"\bNDP\s?\d",
        r"\bNWP\s?\d",
        r"교범",
        r"근거",
    ]
    count = 0
    for p in patterns:
        count += len(re.findall(p, answer, flags=re.IGNORECASE))
    return count


def format_complete(answer):
    required = [
        "1. 질문 해석",
        "2. 관련 교범 근거",
        "3. 작전적 검토",
        "4. 권고 또는 참모 검토사항",
        "5. 한계사항",
    ]
    return all(x in answer for x in required)


def has_limitation(answer):
    keywords = [
        "검색된 자료 범위",
        "추가 확인",
        "단정할 수",
        "제한",
        "한계",
        "확인 필요",
    ]
    return any(k in answer for k in keywords)


def has_forbidden_internal_terms(answer):
    forbidden = ["chunk_id", "chunk", "청크"]
    return any(k.lower() in answer.lower() for k in forbidden)


def keyword_coverage(question, answer):
    # 매우 단순한 키워드 포함률
    q = re.sub(r"[^가-힣A-Za-z0-9\s]", " ", question)
    words = [w for w in q.split() if len(w) >= 2]

    if not words:
        return 0

    hit = sum(1 for w in words if w in answer)
    return round(hit / len(words), 3)


rows = []

for model in MODELS:
    for i, question in enumerate(QUESTIONS, start=1):
        prompt = build_prompt(question)

        try:
            answer, seconds = call_ollama(model, prompt)
            success = True
            error = ""
        except Exception as e:
            answer = ""
            seconds = None
            success = False
            error = str(e)

        rows.append({
            "model": model,
            "question_id": f"Q{i}",
            "question": question,
            "success": success,
            "seconds": seconds,
            "answer_chars": len(answer),
            "citation_count": count_citations(answer),
            "keyword_coverage": keyword_coverage(question, answer),
            "format_complete": format_complete(answer),
            "limitation_present": has_limitation(answer),
            "forbidden_internal_terms": has_forbidden_internal_terms(answer),
            "answer": answer,
            "error": error,
        })

        print("=" * 80)
        print(model, f"Q{i}")
        print("seconds:", seconds)
        print(answer[:2000])


df = pd.DataFrame(rows)

output_path = Path("model_comparison_result.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("저장 완료:", output_path)
display(df)